# Initial Implementation — AI Act Classification Uncertainty

**Team Synthetix** · Avila Kumar Rajaboina · Manideep Galipelli

**Unexecuted, self-contained research notebook.** All source code and draft input data are inside this `.ipynb`; only a separate README is needed. The 19 rule identifiers are experimental and not legally validated. No independent gold labels are available.

## 1. Research design

**RQ1:** Which missing factual details most frequently change an AI system’s regulatory classification?

**RQ2:** Does explicit uncertainty reduce incorrect definitive classifications compared with binary rules?

**RQ3:** How accurately can rule traces identify the provision and factual assumption supporting each decision?

**RQ4:** Which legally ambiguous cases should trigger specialist review rather than automated classification?

**Scope:** Selected provisions of EU AI Act Articles 5 and 6 and Annex III. This starter measures descriptive software behavior only; legal correctness requires qualified independent adjudication.

## 2. Required imports

In [ ]:
from __future__ import annotations
import base64
import csv
import io
import json
import zlib
from collections import Counter, defaultdict
from pathlib import Path

# Standard-library-only implementation. No pip installation required.
VALID_VALUES = {"TRUE", "FALSE", "UNKNOWN"}


## 3. Draft dataset embedded in notebook

The following code contains the complete 140-case hypothetical benchmark (70 A/B pairs) and 19-rule draft codebook as compressed UTF-8 CSV strings. The compression is for packaging only: the code restores ordinary CSV text entirely in memory. No `data/` folder is required.

In [ ]:
# Self-contained, embedded DRAFT CSV input datasets.
# Compressed only to keep the notebook manageable; decoded in memory.
# These are hypothetical research scenarios, NOT adjudicated legal cases.
CASES_CSV_TEXT = zlib.decompress(base64.b64decode(
    "eNrtndtS40i2hu/3Uyi4tgkwYKD7ykm5ujxNAdtQU92xY4dCyAlWlyy5daCa2TER8xrzevMke61MnWykKlwlCZn652KasuU8LOX/aWUqcy3bCqXpzHp31sJxH/mvpeUE2X8frMCxvKgX2tKjP30zkn9FPceLpDeTM3MZB0s/lD3L8+RfpuM4phVIqxf6cWBL80EGoeN7XFgYxbNH044j/+6O6rKj0Pwj9L2ebXkzZ2ZF0gxilxuSfj63vHuuIJAzx6bvVQnSi+gT/8HhYi3XdK1b6fYC+eDIzzIw99c/GCQfWLM/Yl3MLPnk3ndnpr6O/vNn7FA9+WXc6DCyojhMyjI9P5JhbgXb99gQ3LPkgiV9HDjRY2/mqG5Hj+o3/3W1t2+Oeu9HF5OrD+ejm8nlRY8+6o16OyPDjsPIX8igH8rgwbGlMZoYcShDYyZd51YG1F73kf5hy2VEpRps9oCMJ+mXdFXk0yd3biw9+sTiXwZ8iW9YtvqFd29YnkEGl8GucTOXxp3jSioutAMqPaSfLKgGur8u16FsaswcalLABjBu48iY+XQd9cMIZOi71ITPcxnNqR76P0NZiK5+NOZWsKCP6KI52dVwQmMho13j3ePSpwsjMilXsXT9xwXdQiOx3k/GIvYc21nSt0vH9aOe4S9Vr2fG7SM1bxnfuo5N1rGpFoNuvB0HUpUQSWvxMzcilGlpVEFkOS51K5CqyVZIBuGyyCJ6OKlWu/Ke6rNd+tq5S2727k7vLLkX2l4GtSOMLTZJr3c1vngzufjFvHz7dnI2GZ2bN+Pfbsy/j6cT+re+pR8ursc3vZ3/29kJH8NILkwnNC1nZ+cn+uTBoltEf+7s3Ew/jHd2ejs7NGRmfNdMGjL3jqe+fPf71eXNu/ENFXlujj7cvLucmtc32ajZ2fkn/dDxTDIZj06bTFV/+QvLc5axa/Fwox+b2dgj3dtzz/kzbqRSPQrNfPCZhAUzdO49dYtI9TzC1mv+cPHrxeXHi++o/J/00//Z2RkdmUWB7uz8707vuY3qqf9dXN5Q0aPpzfhNr1itcTU5v7zZNS780nFn0LBOYWVYS0ab5e4agmTIozA07gJ/YYwM3yMOOJ7xhIo/GymSDKUBUl5IrZKzXWM6fju5GL8fX9BQ3d//yThLZMJEJ1HcFasmZrtOuFD1/EziMTz5OWlwVhW3lalpKIBm6trtnd38tjfc772b/PKul1QS0w/Vk0N3Mik1Yhw59NxY8sPDi6hPkhhCGg/n1CJVH32WQ0kZge+qoqh4SlEBioKioOh3UfTt6Px6DIb+CAwdkCd6fanEd312OSU1EkUH2hdNKEFcZOItafBS7xbGvfQUS0ICxIJcfYc6QaohRpHEuSEODRpqamLmWzm3HsgRJngGfhimVAl3jY8J8qh5BeSFvk2t7HOJXC1dPnO4v5o/IfU9vGNLO5qfM2IXk4vvzVexGC64LvLLH9jy+RVP+JhckT4+1ARjJoOaycj26Kdl5YYzkr7/UIDUt91Mum7mt71JF2915CtAVbYDRFJEGjRPJFFGJAEigUjdJNJ3VgUcdRlHB+Qgjd980GPbHL+/TGaaB8pH8nhmSQZ0U8PSvJHtl6xMhv/5178NufA1LWax5gffClmgDQuS5pjKiFHg2NxANhO1Oin6lqaXC2UPl80RWncyejSSldZv4k4g79V66ReQk12iP/ZrZs1Ym0VbTM20eaDxVYHvL8LeeBbra38k5iRjxcyMYsrUDOZnP/hEjzy7iYmiHls8MdVjy5R/qfmp7zXpfiVq0hPD6iYAdQp1B82jTlSgTgB1QN0Ph7r6lsEAuk6B7pB8uo+X01+vzkdn4wLoDhOfTi4YFNQXTQxe0I983yWJW1GKveQamTNP2ffBV7MzK6J6vVC/HQgkee+zfMk/XErbuXvMlvytEuBlY0JzkRn6VdDN44Xl9R2vT1f1Xd9fGhH3fR126rL8xcTC+iTb4J02mKLvOP8TwINvB+QND5tHnqhAngDygLwfFnm1rd2Bd53i3RG5eFfTS5rNkqVHb8c3vxPsjvI1O/WiwGOWECj4JQITLwHRXezZiZjVyrtPY5aYQ/Xc8+sA6gpNPrVJojmZ8MG59yTRj0lxy0B5oGYo8gTSlQ8WVTLizZDGhBvthK6GhgJlYdZLZQWz/tIKqAkEJW6UEz321Qw6VNhLdiWqv6ne2NNYdZ41Db61Inve570eVBxPzpNXCeuspAESBXR//0EfFK4mTkT1IvNaWzu/E2RumojzJgldNZv9h3rjkPTZTLbOmslIqr8iNdRMNdTMfKhlu16b8wuHZiJKxcmvNAOsVKw86lH3L96Mpm8a5aUo46UAL8FL8HJDXtbkVAKWHYXlsOBcnl1e3Ewvz4mWw9y71LIywni59IMoTEGZTkBD/c7jqXqLLIzUNPprPCwiLgFsCRSNAhSTVyiF3j5nP7F/y1ucs33S6xBMvi+8HqqXfdMnllozKbgHP/EHR9+wHfSJUvQJoA/o+4HRV9e7YoCvo+A7Jp9vOj6bfpjccL8IesfJiV2aj1neoz5fRgDkY2Jk5pCnn3YQOxHjRBPLtsKnr0ayJqweAdMY5VlrmFiMJn28nVl9q+exdrGKdI9MaFBhjAu6Zwk7J5NncM52/XjWn/vqt1Vz3eI16fbrunGX9yh5iPygL0/yA/TZsoVJg6FJp2t0cTH+zaTxouhT0QBQR1HnuB3qiCfUEaAOqPPS1KltiQvI6RhyTlYOPIzevJ9cX+u9Iifa4aHhl4QzyTaLEIDIcA5N96zZwgnVJgt+v3/n+p81hKiHNFuR2aSlv7A+8TJ59u6A5nvqcDTrNQPIN+33pcKpNmoGQcnxqN+unjWuYcXTUzre9Tu54YLIy+aC6gXLh9xYmWnCzDY/5s5fODadpsxJO5QRlZQRoAwoA0cGiPk+xJySI5PdFHP829n4Ktn0epq/qVObXMkO6SYFktsdn8j+7ERz3s5QmM2og0tqMmV56sJkJwTNUMimgf+ZWqLGcrIPYBYH1KTICj/l55+cKGusukdJkCHeTJHtTiXL2WRiPs5eiIpiBE74iUy/sIg2G25lKK6MF4L9PEEVXaxR9UDmrvsQ1FVuk+L8MYcUpluNUYp+qIeomY9Mk0dmA6fmC4F8eMjyJtgs6E824Ov39fS7AQ7EFZqeFakecqypkvP69a3V//eH0Xk2JvV5/Wd1H48B9Rg4bXoX8Cl5mVWPAIFHAB4BeAR05hFQy7th8B/8z/m/v6c36r2dnKswUvt7T7boVXM/h/4T1lP3iKSWCg7FElI/8ePoKayp+09DgWbDYOWhkI5eIxm9RjJ69c7ljdYmFrEbOX0VBqs65oEnIwYvv9zhK8O6UZ+bKDvzVdiLgzWJRlFfGKGdgvz3oncDyte5fPyU81WtANmZ7Md7zZNdrJFdgOwgO8j+A5C9tiV7YL1bWBd7+5wZ5e8fzi/G05Egtt/83uMPyW0fUfMZW8UQFTokmdr2xFxWsSoiK7iXkQrqyvDnsBV/0U10qAIVt9+OI47X71keB9bkAPuzcO4sFej5Utd3Iv2KkO7TYhkRPUt2SBVWcJJA5mEeVpZuD1+qbmfsJQH+6V5YdrTLtlPtUsX70eb7t5MOmta95CDq1q3j8p5gjjXg244vqeP+wrHNh9jlqLj66wZCOxRM1UoM+9S8hcjxTcavWRmEq6HrS1rylA87b6ajtzfGvPhET1W8a5ylOY4MznFEQzuRgH4oq03tFj+GrfuwpzRFnVTXz2iMShrD6gSs1i6XEO4a5+of7NHwQFNBe+M7QhCP/McEAumZiRnH+iVB7yXx5S/oK+6M49lR9vOcCToR1M9Jfakd+G2zahr9i593Wr6iTL4C8oV8vyrf+uLtQbzfIl7OBXE2nbyfXJCFp5PrX1m8A/Xstfn6pe9ypCc+2c7DX2/71SpW0daTStP8OHn48P/869+k6MBZOCrGJj3v6Ss+Ue+7nIdHuT/a0dIN014YXfFzehTMsTikEnkZ2VeG5fLW5ESsyg1hq2qVTvLI5Vm1atb2bXHM856YaXGm6gWNUOUhpx2vPb6QCkPORjKzjjMqCsYyb62wiZOYXHNyhs9U4bPMfIqn6zRpNuXf/iFtxRA9kPgemCqXX3PiXhmiSZD0Z5hoW1U+qF3lokzlAirfOpXXUC0k/golzjkL3o6SXBLT0RUvkfLH+lGenByyfdelGxsmSVVIoZzdjrxx7hMPisBiAsz9yL8PrOVcpcVb+iqJGWfF4yQs+kW+f++pnBQGmcbiG7HqcycVqfVRXvDUHrDOl5L717u9t9UFGrex4842l7Ruo1ko0kyLNNUGC/6ERhJ5w5YXNiNsm4wnQ1NvA5YRV2fb0YPpLGhshfVXmBvYVHWT1ZqcJ6+NMyXWkiZsqzQPapemKJemgDQhzRrjkUKXG+mSQ8KLyeX78c10cmbSaB3/cjlVq1ccFH7kJQpUubECP90elwa5S06B5HnGYhUD7tbxF5LTXBgLaYXJlrvCihQfZUlLUAVwgtRQ0nBPV7AiUie/mGPBh+uqFFnxWTOsb1iRylpprhbTxEs4ft1hZj00db8amby61ue7mNPBfzYlB86wlfEVY0KSunq9oRzJFtI/PB1YSo4V5thWSR7WLklRKUkBSbYqye+vCHrcdj0eqQg47y9vxrkBWY0qqvaVXjVaupZnuKyUXGh8r6LCy/A7Y6leK4f928ckZKxyc43wz5gstOqh6p0t6ne2HQehDnrKm+qdhVxVX49+qCJxxdGcBsY/ko0pZLQ8lWc2uMJkH/3KNvqd3lWSGH3JueBXG76hhAOOfS/NXMmrpTWwMSSZK5j5XMFU/WhEyysibmSjC99lk++ymY2BZla508GRk0dlJE+GUWg1/aZ5XVNUqEFfZMdW9pMP1o+YlhhoW1F1VDuqRAWqBFAFVL0AqmpYrQCnXh2nhsVVh2sm1FCt0ZPjyT5olo5ybc9MYRLDQerVSza9sUXv92M5V7HMuyecFeY7q1c5Yfbz1YnNVfrbSlr29r91jqPb3oQsFde+TLN6fIhsyvDcmVtdk5kvqy/55Ok+11LTbKsqh7WrUjxRpYAq21RlDY9LSPIVSfJYb03T1p9cvJ2OWJbH6mH5mX9nxJHaKVkMFhX5VKbH/dWX8GYH9zH309f2hqZxAFToS6uQ36aQcWVVfh9VsY53F1hhFMTUPTKy/lFvsJnu7MBR99JcLc1ULTd1yxsQic5Ol3WwOfcxE8VgI1GsN3Bb9XBcux5EqR4E9NCkHmp6Owsx1CqGlYilrIMT9VwI7TmP4FUB8HGDe88Io5ifs2rjhCutwGNvbRnwdorFQq4Jgfco0KfeTHtjM0lNWTge202tX3Ah+UnINU1kpyHZTvryLHHDwWaayHOnNuWb5V3LE7WSBnW7W3g4HGykhy+2dlvFcVK7OMS6OATE0bg46n5SQBm1K+N0LaMHf6T3+mTJ0AvR9cO5H0Qc4z7N9PNk24AfR8u4KrhXMd5VngtpTQ4rUejT+niT1uGGWsjiaJmFes3k/H9zh7zcx/x8eFisuoVHx+FGAvlqi7dVJKe1i0Q8FYmASFoQSd2PECikCYUMOMbY+PqaxKH2246nf5+cjXnVeLCnpiHJG2PqGzdq1eEK7UBKr5g7TufH46PAlpudjbZlUURFH0u6NCL0GeF1nVzrMm6lJ+/oR4Ure0ebKUX3wMxb05xICo5L0nCz0PAWHiNH3+pnlTR3SxWyv1e7QkSlQgQU0qRC6n6GQB71y4ND3pyPPprji7eX07Nx4mYNdNCb5DSuRdM3mtaupfvi+bK+sZE+nuM6ScCIL6zkqm1M2XlZx3uQ1If78ul5MmiKJSdNccLecDONrO/EaUwh2eHdlb41m2EmE8hwI4FUN3VbxbFfuzhEuTgExNGsOOp+dEAZ9SqDo7W8n/wyTRd2BzpSy60fzDjMnHamCpKQfGu5iiiwHqTbT+NOqpAJK95TMf2jZTw4ocVR4XTB2WmbPObl2gmYpP48HuTxZlJYOPf6daSpa2xOC9wzPvOZ1JP2qIWnxPFGWqho57YKYVC7EMS6EASE0IAQ6n4iQAU1qoBjfvztA1n8bGy+Gb+/PJuOzviw5CCN+hEHkUoO51FtyTmFpeP6USoLGrnSCuy5zm9CDsmKFPTUNjT+iKllauadz8h5FyNdr8Ms208PRf4t/U1WRT5R7p1sJoq0/nTbu9oP0ZAskj6beZ15n1t4SJxsJI8vNXZbNXJQu0ZElUYENNK4Rup+fkAgdQvkcG0n1eAwSZhS2EWgAsElYcNILLEb9bNNIsnOkYLTo3rk8Mx2oXYQ6m0nBY35wWrq1jhJruUsltSRtMeJAZ4RIV8FMus7Xp+u6ru+vyQ3j3q7HiNfXZb7bgvrk6w52dX1E1sUTIvI+M2cw+l+8sGGslpXY+8LiWh/H1+vQo9+yT8cnRvT8fV4ND17Z1yfjS9G08nlzwZdaZyPfxmdn/9ujN787cMbrmb8Zte4spwgi2W/FsQ+x1CKODXMeoXjrT2ldwXSjJyp7P8kkqk2ZwBoK+B9QtjD5jPVKu6Kde4KcBfcbYG7NeTtAHVB3W2k7tG6t3v0FW9XH9HrFHZv+ahpX6X0DFWAKzLkg0NzxjXu0tSFqnedf6SpafXVHOUZ+IXbCwADwEUAH7UEYLEOYAEAA8DblZEP9AV9t5G+w3X3d/gV99cl48ZkqD6/KCFedYC+C/+W46ipCN531lPqJt8XlvoBW3i74C14W+TtsCXeinXeCvAWvMXiLmgL2tZN2+N17/b4q97trZ+kte8AaG3Xj2f9uR8yGquWFIrXLF0r4uaBt/BvQVwQt0jc45aIK9aJK0BcEBfLt8AtcFs7btejHg5OvrZXd0kd8AiAMpw7yy4g1/f6VNQiSdVFXXddvR17FbueXm/g/kxuuCArUBFoAF64umAv2Ftk70lL7BXr7BVgL9iLZV2QF+Rtkryn617v6Ve8XpsamWdS6QB4o7kTzPrM0Ue1ESzJQfuEu3Sx5q7OawvgwtkFcoHcInJPW0KuWEeuAHKBXCzsgrfgbe28Pdhbc3EP9r7i4kbSnnsMv77tu1R4N3aKxW7k9EOHjJp/vf4+zZPRZz/4xMFR+MoQxIWTC+gCugXoDvZagq5Yh64AdAFdLOUCuUBuI8jdr0qccrCfOLzE2zSm3mcnmhtzP+ZDs/08NN5KeocCeNki1DCHxho1PdSUTU4l5KHXHE7OzffB8h4L94aPGVt0N2MvjY32LPTS08BZspU5YuA6dHWGiH4obU4cXliEoAeJVfMWsnHBJsnzqszQgDE8YOB4C3C83xKORSWOBXAMHGMJGCwGi1th8aDSNR5UuMYJ0Epymb0sh8MF9ZOs6zxYX1yUSK9IDl4wklkMAVAMzxg0Bo3LaTxoicaiksYCNAaNsWgMFoPFrbD4oNIzPqjwjG1uf4cYHMh7/TaxGr/ZJfpjH9yFFwzygrwV5D1oibyikrwC5AV5sRQM7AK7dWP3sNLhPaxweB0vpFHWqbdxnUkOBADD9QWDXw+DD1tisKhksACDwWAs+4LAIHCDBD6q9IKPKrzg0Ockz/0EMN3BcOfyBQHHcIkB5NcD5KOWgCwqgSwAZAAZi8SgMWjcEo2Hle7xsMI9lgsZ3NPPH/uWM+vQsY0Xzi8E9sIZBn5fD36HLeFXVOJXAL/AL5aGAV/Atxn4Hlf6vsdVS8M6RE3/PqCx3h34diQBERgMFxgUfj0UPm6JwqKSwgIUBoWxCAwEA8FNIvik0hE+qXCEI0JvyGjpZAifTmUoAo/hFQPJrwfJJy0hWVQiWQDJQDIWiwFkALk9IJ9W+sinVRslPBncP3Zv21onchoBwfCKAeHXA+HTliAsKiEsAGFAGEvFIDAI3ByBD/eq3ODDvaooasSmvmXzqYcuBXl/8XRHgC88YPD31fD3YK8l/opK/grwF/zFIjDoC/o2RF9OPHc2neibPrl4Ox0xeQtJ57Ikn9KVdhQ4tqKLdSejx34gXcU2Gt+BFUZBbNMAlAnz2MBkNHkfuxw88jE3sW4q3zAiMMHOVj3j12TUgRkn/tyupEaT1e6n6F0zK7ALtxfg3QLw7rcEXlEKXgHwArxY7AV1Qd0GqDsodXcHJe7uZy6gC7ztVp4iIBe+Lqj7aqg7aIm6opS6AtQFdbGwC+aCubUz96DU0z0o8XTvrbALxH35jESgLHxbcPbVcPagJc6KUs4KcBacxbItIAvIfj9kD0ud2cMSZzYgLnWBsp3JMATYwqkFb18Nbw9b4q0o5a0Ab8FbLNWCtqBtjbQ9KvVuj0q825kT8k6wqD+X1ADOz/Py5O1cziAQGB4vGPxqGHzUEoNFKYMFGAwGY4kXAAaAmwXwsNQJHpY4wbeBM7uX/YVP1KJR0w0Cv3S2IAAXTi+Y+2qYO2yJuaKUuQLMBXOx1Avigrg1E/e41Ms9LvFyaZ5+RwO8z6gK/E68Y+tIOiBgF64uwPtqwHvcEnhFKXgFwAvwYlEX1AV1G6DuSam7e1Li7qpY5h1Abaey+gC48HTB3FfD3JOWmCtKmSvAXDAXi7ogLohbM3FPS73c09KgYkSCzsS46UTGHvAWPi6I+2qIe9oScUUpcQWIC+JiGRe4BW5rxO3RXpmDe7RXtmuBg5X7i/6tZX+Kl90IV/7ieXkAXLi4YO5rYe7hXkvMFaXMFWAumItlXBAXxK2ZuJwK7Wp6+ebDGd2M0dvxze9M3EIqNILs0veYdA6/qLL9gJiSQJagZHHVcajizvQD/5aaQ+NxRgTaNT4SLudkjGg1DWVxPYC4x1u2OMtP4MdkEydM78CM/+burfZto+yUnUnac6Vtkjynej8Sg5PhYCY4MJXS5Kz+itS4MtW4MvNxZabDqRH2tvSAycGbaDXDbgHEhNyvmADofR5691tCryhFrwB6gd5Xjd6tcezB3VfI3UGpyzvYwOVdqO64/ZlM0990g7rdyt8D8MLnBXu3hb2DltgrStkrwF6w93Wzt4ZFZpAX5P1W8h6Uer0HG3i90pUPVjJf7wZzXz6rDzgLHxek3RbSHrREWlFKWgHSgrRYxgVmgdk6MHtY6tAebrKMa9lzx5PBY3c425mMPsAtHFsQd1uIe9gScUUpcQWIC+JiyRa8BW9r5e1RqYd7tIGHS8KKpB05D7LPplsq9HUGvZ1L4AMEw+kFhLcFwkctQViUQlgAwoAwFnpBYBC4YQIPS93g4QZucOR3aMHhpRP4gK9wcoHYbUHssCXEilLECiAWiMVSLgALwH4fYI9LfdjjDXzYBzl3bFf28ws7g9uOZOoBdeHYgrvbwt3jlrgrSrkrwF1wFwu2gC6gWz90T0qd3ZNviykWSi/s0kmITuXtAXrh8oK+20Lfk5boK0rpK0Bf0BfLvGAv2NsYe09LPd/TjXbsyjCMA9nXfOgOeTuRxQfEhb8L5m4Lc09bYq4oZa4Ac8FcLO8CuABurcAd7pU5ucO9jZZ3edRxkzoXxrEDWXwAXDi5YO6WMPdoryXmilLmCjAXzMVSLogL4tZMXE6M9vFy+uvV+ehsbI7fXyrR8xe5o+t4yr5y4Uc69CEVH6k3UwQs/+6OKcvQWrqWnZFq17ghDhESqUlcbRLQnDut6ZPfByarRe0KlzGT7C5NTBmGsdyuJDwfMyskxtK2Y2P/ULBNem9mvTclwVdZycxGSv3VJmOMk1LqMWZmY+w1uLpHqT4rwPuF7gO6z4Pufu/d5Jd3zQJXVAFXALgA7msF7tYt5oK2r4C2gyr3dvBM95bZyEnUXxC33Uq5A+LCxQV0txi6gxagK6qgKwBdQPc1Q3f7lnOB3FeA3IMqP/fgmX7uZwLT3Kdh9pLQffn0OwAtvFugdotRe9ACakUVagVQC9Ri6RacBWe/n7OHVS7t4TNd2tCe+77bp4/u7l4StZ1JvQPiwrkFdLcYuoctQFdUQVcAuoAulm6BXCC3VuQeVfm5R8/0c+d+uHQiqwPQ7VySHcAXHi/wu8X4PWoBv6IKvwL4BX6xxAv2gr3NsHdY5foOn+n60h12+wpnL3sg4oUT8YCzcHOB2i1G7bAF1Ioq1AqgFqjFci5AC9DWANrjKp/2+Jk+LU3NvXDpB9FLYrYjOXdAWzi24O0W8/a4Bd6KKt4K8Ba8xYItYAvY1gTbkyrn9uSZzi013FdnCV70cG+n0uuAufBxgd0txu5JC9gVVdgVwC6wi4VcQBfQrRm6p1W+7umz9+VG1PAX5G0n8uqAs3BvQdotJu1pC6QVVaQVIC1Ii8VbYBaY/V7MHu9VOLTHe8/dbUs2jIJYZ9dReWxeNAj5i2fTAXDh2oK528vc4V4LzBVVzBVgLpiLZVsQF8Stjbj/D/KO28s="
)).decode("utf-8")
RULES_CSV_TEXT = zlib.decompress(base64.b64decode(
    "eNrtWsty2zgW3c9XoLyyq+h2x46dTFRdXYzMJOyWJY8sZ5IVCiIhGQlFMAAoW1W96I/oL5wvmXPBl/zQ9E4q13hlhwQvgHMOzr0XjikzyVUaZHIuMl4YvVRW6TywujSJ5Etp/D/1bKYShRGlyQIjf5TKyBTDZaoS4aQN5F0iC4eh6w8Tkacqxa/cSFtmDl8ulbzlmVooJ2i0rSeuX1g8Le8/k6b5igL9IzzlF+EwvrwehJN4NAxC49jp/quDfXEQjH7jxz8fv+ajcfwxHoYDPhxNeH80vBoN4vNwEp0HN84V9t3RkSzNYSbvfsJPXQj8OPJTHiY6dzJ3R9HwaPJlcvRradQv/WgQfXl3QpHHr87e/jNYiFwVZYb1LyXXhqfSbx3/cDK5ydWPUvYWWKshvFJlnTYeGIDBrZrnClCK3PEbYRZBcDkefYrfx1gd74fDc7/QIJzPDRbkZMpoFGsxZRrIWUZRVZ44lojSeiCPsArPHKvZWWAftsf8vliFIMPKAaI2q+B8HH6Y8OtheIn5PwOZgJC9GvVjwHbVB4LDjx22ycG+OjjcV2prGFvt1WYTbVQ+5/gmVZVengZs7wrDMskSvSh0madriC2E/W6ZUwsZsKmwyjKgwFLpjCKMgI4zUjj6vccWpXV458NYubcBp+hidF98s60BIxfai0nlM2lkjiMq0zLxEuC32nwvMpHIYOE3n5E6rZhJt+Lt+Xwav6h5XSEwlWwmMit7wGKmcg890zNWz87a2T2WitadQqtFaQi2RoK17J4E8Yzj9/Pr/sSDeFad4MP96UGPhXku71i8LUTheUDQ4XhiWq6gNZieTHvuRhlYnDBAD4Fm2uDorXhjfkHwKf74iY/jq9/XcLzyaFcyzGnqOjqDHToPVr07ZmVCUB7aAgcXjsCEtdJakiHLtWOzMstWDBhrILsBwnA4jL7wOI5rEI879OKt4VfjphSvt4LEUFr5ND7RnQAY7RqZcnJRweI3cHIAwRk993puZaQgroIUBmg6lDZg8q/rcBB/iPtdeqCwXlzpwY4x6a3bv1H2Ox3QNlXgUGUlHSqS5Exl0vIc6dBQXkYaJvOLvvSjS9oYH0ef4+jfwd5QGKNvmYPJQWsScvUG7x8wkUGEgM+WPn17h2sRmR4EMHYmptY/F1nGOpt9cIQ3GeHn68EwGofv40E8+drZ4XRrQDth5tJZLuaSUq2YqowOKfkeMoiGkefImAlfllkuTf26J++KTPsK5OmkPZU3YqkotbfZe1PeidpQGMPuTcPmhlJRwLooXuprIqiy+x8MsSejydfLiI0ugScxDBVf+V/YaDj4+o6REWFtdUJfS4istXa7bjDN4Wr8hRZEvrIUmS/J0jrZ0TNIEwerDo3MKGF1ud/ST5uY74/jC88rHfCO+e0dMZiCWqq0BF0JMjnWi3Q3m/mU6I9WVQHQJnpWZzJb8epYUT0BaqsjJbxcUBbAY21ZFKDJ8psShRLvnKZ6z5Fh9fQb2fbSF8ZAVUxRPc9gaXZTWeJnZu3MR2vTsqVlbcBDH4X5qddTwXPUxoewLiPH4eW9OlJuTR1AU/kWItHzqnyh9kEQkZCLrOolqABOIHLrNZIYUcByqZgxuXT0NknckqsFzMX2yrzyGgjBDwWZmzzhuh3KmqEe/WpRrFlIWy49R47fx6OLaDKO+9j4JPo4Gq/Z/3xrNE+VXlAhn3Aq9edoFqpuqOcLVMuR2qyqXN4INE5BJm5RWHH84JJKusT3SV4bFpxnYioz7xB/Uy/vXTWRD+uZVw+K4kQYvNQlDvVz5HdMPU7U0dyxe4Mi8xSl5uH+m63x/JAvqqWKcpqpBLYukgRuqciKbYHup4fznXFq+LiG3VaCMNS6SN4JRlE56dOwZ9nfPFDl1DHvG3ZRuptWVpu0QAEdbD6XtBIBLXScBexeiIpDasrmpTDpcxRH03TwV0E3XaFhnOxVU1+SPE52YAN2ZbHvhu0HHD/Zkey9bz627D9//uX7Mm0YycwXbE+U0AtRtMh2HcwaE8+Y0+NHnB7vhlMUdc7fYcBWjbDOoIsujeS3VKRzqtWyVa++3Gjb7Q0U9+tY7H6sF75jfvKI75Pd8N1dY9VnuNkGarL2HeRQuf0GoqNmoAdyqZuvGKX/nOqwF8r560eUv94R5Qv0ziuf0FGom1L5S1juv71zbTuOFN9ejdj1kZtE0Ib12NKtKGpBuvxGIe8fv4iAnz4SweluRFCVcdRtK9Rg1L03/K8ZwFTmcqYcl5maq+p2ZRP5lq6zCG8rzVJBMR7bOsBLigf1Z4+oP9sN9Q+r+ob49j5H5UtpnaqugzdfKu8NxC1bC/RCcszfPCL5zW5IXqh51YLxqTYpSreG5SU6IrpkqR83f7/cQPFFEwZ4rLKyoqr6lDB3RmcvtMf87SPa3+6G9m9lWv2Hgbr7rf6I5omvrN7ybghq+f/do4UpAMBHtQT0jH0DVqq+aknR7SX0KqGLVioP5f+x0f8Xn4WjAg=="
)).decode("utf-8")


## 4. Data-loading and preprocessing functions

Decode the JSON fields, validate the expected three-valued facts (`TRUE`, `FALSE`, `UNKNOWN`), preserve absent keys separately, and parse rule predicates. These functions do not assign legal labels.

In [ ]:
def load_csv(source: str) -> list[dict]:
    """Load CSV directly from embedded notebook text (no external files)."""
    return list(csv.DictReader(io.StringIO(source)))


def prepare_cases(source: str) -> list[dict]:
    """Decode JSON columns and preserve absent keys separately from UNKNOWN values."""
    cases = []
    for row in load_csv(source):
        case = dict(row)
        case["facts"] = json.loads(row["facts_json"])
        case["candidate_rules"] = json.loads(row["candidate_rule_ids_json"])
        if not isinstance(case["facts"], dict) or not isinstance(case["candidate_rules"], list):
            raise ValueError(f"Malformed case: {row['case_id']}")
        for key, obj in case["facts"].items():
            if not isinstance(obj, dict) or obj.get("value") not in VALID_VALUES:
                raise ValueError(f"Invalid fact {key!r} in {row['case_id']}")
        cases.append(case)
    return cases


def prepare_rules(source: str) -> dict[str, dict]:
    """Load prototype rules; retain legal approval status without assuming approval."""
    rules = {}
    for row in load_csv(source):
        row["required"] = [x.strip() for x in row["required_predicates"].split(";") if x.strip()]
        row["exceptions"] = [x.strip() for x in row["exception_predicates"].split(";") if x.strip()]
        rules[row["rule_id"]] = row
    return rules




## 5. Predicate-status handling and trace construction

Each candidate rule trace records its provision reference, official URL, review status, predicate values and hypothetical evidence origins. `ABSENT_KEY` and `EXPLICIT_UNKNOWN` remain distinct.

In [ ]:
def fact_status(facts: dict, key: str) -> str:
    if key not in facts:
        return "ABSENT_KEY"
    value = facts[key]["value"]
    return "EXPLICIT_UNKNOWN" if value == "UNKNOWN" else "KNOWN_STIPULATED"


def rule_trace(case: dict, rule: dict) -> dict:
    """Record all relevant predicates and their hypothetical evidence origins."""
    predicates = []
    for key in dict.fromkeys(rule["required"] + rule["exceptions"]):
        fact = case["facts"].get(key)
        predicates.append({
            "predicate": key,
            "status": fact_status(case["facts"], key),
            "value": fact.get("value") if fact else None,
            "evidence_origin": fact.get("evidence_origin") if fact else None,
        })
    return {
        "rule_id": rule["rule_id"], "provision": rule["legal_provision"],
        "official_url": rule["official_url"],
        "legal_review_status": rule["legal_review_status"],
        "predicates": predicates,
    }




## 6. Binary baseline and uncertainty-aware prototype

The binary baseline is deliberately closed-world and produces provisional software classifications. The uncertainty-aware method requests missing facts or escalates unresolved/unapproved rules; it does not certify regulatory status.

In [ ]:
def binary_baseline(case: dict, rules: dict) -> dict:
    """Deliberately closed-world software baseline; NEVER a legal classification."""
    traces = [rule_trace(case, rules[r]) for r in case["candidate_rules"]]
    return {"case_id": case["case_id"], "method": "binary", "action": "CLASSIFY",
            "reason": "CLOSED_WORLD_BASELINE", "traces": traces}


def uncertainty_aware(case: dict, rules: dict) -> dict:
    """Conservative research routing; all draft/unapproved rules require human review.

    Precedence: absent/unknown facts -> request information when there are no
    known conflicts; unapproved rules and unresolved scope -> escalate.
    This is an illustrative routing algorithm, NOT legally reviewed logic.
    """
    traces = [rule_trace(case, rules[r]) for r in case["candidate_rules"]]
    statuses = [p["status"] for t in traces for p in t["predicates"]]
    has_unknown = any(s in {"EXPLICIT_UNKNOWN", "ABSENT_KEY"} for s in statuses)
    unapproved = any(t["legal_review_status"] != "APPROVED" for t in traces)
    if has_unknown:
        action, reason = "REQUEST_MISSING_INFORMATION", "MISSING_OR_UNKNOWN_FACTS"
    elif unapproved:
        action, reason = "ESCALATE_FOR_LEGAL_REVIEW", "DRAFT_UNAPPROVED_RULES"
    else:
        # Never silently make a legal conclusion even if someone changes the CSV.
        action, reason = "ESCALATE_FOR_LEGAL_REVIEW", "LEGAL_SCOPE_REQUIRES_REVIEW"
    return {"case_id": case["case_id"], "method": "uncertainty_aware",
            "action": action, "reason": reason, "traces": traces}




## 7. Experimental design and evaluation functions

Run both methods on the same 140 cases, group 70 paired cases, compare routing actions, audit structural trace completeness and summarize outputs. Legal accuracy, unsafe definitive rates and correct specialist-review routing are explicitly unavailable until qualified legal adjudication.

In [ ]:
def run_prototype(cases: list[dict], rules: dict) -> list[dict]:
    decisions = []
    for case in cases:
        unknown_rules = set(case["candidate_rules"]) - set(rules)
        if unknown_rules:
            raise ValueError(f"Unknown rule IDs in {case['case_id']}: {unknown_rules}")
        decisions.extend((binary_baseline(case, rules), uncertainty_aware(case, rules)))
    return decisions


def evaluate_descriptive(cases: list[dict], decisions: list[dict]) -> dict:
    """Compute only observable software metrics; do NOT invent legal gold labels."""
    by_id = {(d["case_id"], d["method"]): d for d in decisions}
    by_pair = defaultdict(list)
    for case in cases:
        by_pair[case["pair_id"]].append(case["case_id"])
    transitions = {}
    for method in ("binary", "uncertainty_aware"):
        transitions[method] = sum(
            by_id[(ids[0], method)]["action"] != by_id[(ids[1], method)]["action"]
            for ids in by_pair.values() if len(ids) == 2
        )
    trace_total = sum(len(d["traces"]) for d in decisions)
    complete = sum(all(t.get("rule_id") and t.get("provision") and
                       all(p.get("predicate") and p.get("status") for p in t["predicates"])
                       for t in d["traces"]) for d in decisions)
    return {
        "cases": len(cases), "pairs": len(by_pair),
        "actions": dict(Counter(f"{d['method']}:{d['action']}" for d in decisions)),
        "pair_action_changes": transitions,
        "decisions_with_structurally_complete_traces": complete,
        "decisions": len(decisions), "individual_rule_traces": trace_total,
        "legal_accuracy": None, "incorrect_definitive_rate": None,
        "specialist_review_correctness": None,
        "warning": "Unadjudicated hypothetical prototype; no legal validation."
    }




## 8. Planned experiment — do not execute for initial submission

The following cell is the complete experiment entry point. It will load the **embedded** draft datasets and calculate descriptive metrics when executed later. The notebook is delivered with **no execution counts and no saved outputs**.

In [ ]:
cases = prepare_cases(CASES_CSV_TEXT)
rules = prepare_rules(RULES_CSV_TEXT)

# Basic structural checks; these do not verify legal correctness.
assert len(cases) == 140, "Expected 140 draft scenarios"
assert len(rules) == 19, "Expected 19 draft rule IDs"
assert len({c["pair_id"] for c in cases}) == 70, "Expected 70 A/B pairs"

decisions = run_prototype(cases, rules)
summary = evaluate_descriptive(cases, decisions)
print(json.dumps(summary, indent=2))


## 9. Planned legal validation (not yet implemented)

Two qualified legal reviewers should independently adjudicate applicable provisions and classification outcomes after a fixed legal-source cutoff is documented. Only after adjudication may the team calculate accuracy, false definitive classifications, legal trace correctness or specialist-review appropriateness.

In [ ]:
# Future validation placeholder — intentionally not run.
def evaluate_against_adjudicated_labels(decisions, adjudicated_labels):
    """Require independent qualified labels before any legal accuracy evaluation."""
    if not adjudicated_labels:
        raise ValueError("Independent adjudicated legal labels are required.")
    raise NotImplementedError(
        "Design adjudication matching, disagreements, and accuracy metrics with legal supervision."
    )


## 10. Intended interpretation and limitations

- **RQ1:** Report changed routing actions within controlled pairs, not proved legal classification changes.
- **RQ2:** Compare definitive-output rates; incorrect definitive rates require gold labels.
- **RQ3:** Check structural trace completeness; legally correct references require expert review.
- **RQ4:** Report escalation frequencies and reasons; appropriateness requires adjudication.

**Primary source:** [Regulation (EU) 2024/1689](https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng). The legal source cutoff and rule approvals remain outstanding.